# `news_threat_gate2.py` — Playground

Manual verification notebook for **Gate 2: News Threat Assessment** (Claude call 1 of 4).

| Function | Status | Notes |
|---|---|---|
| `assess_gate2_news_threat(candidate, headlines=None)` | ✅ built | One focused Claude question: any catastrophic threat? Any threat → block |

**Output:** `{passed, threat_detected, threat_categories, reason, headlines_used}`

`threat_categories` is a list of `ThreatCategory` values (8 categories: fraud/SEC, short-seller/activist, regulatory/legal, FDA/recall/safety, leadership departure/scandal, major contract loss, sanctions/trade restriction, pump-and-dump/manipulation). More than one can fire at once; `threat_detected` is derived from this list being non-empty.

**Decision logic:**

| Situation | Result |
|---|---|
| One or more threat categories detected | `passed=False` (BLOCK) |
| No threat | `passed=True`, `threat_categories=[]` |
| Empty headlines | `passed=True`, no LLM call (no news = no threat) |
| LLM unavailable | `passed=False`, `reason='llm_unavailable'` (don't trade blind) |

**Standard used:** `helpers/llm/client.build_agent` + `run_agent` (cheap Haiku default, per-gate overridable). Live calls need `ANTHROPIC_API_KEY` in `.env`.

In [1]:
import sys
import pathlib

# Add 02_intelligence/ (for helpers) and gate2_news_threat/ (for news_threat_gate2.py directly).
intelligence_dir = pathlib.Path('backend/02_intelligence').resolve()
gate2_dir        = intelligence_dir / 'gate2_news_threat'

for p in [str(intelligence_dir), str(gate2_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from news_threat_gate2 import assess_gate2_news_threat
from helpers.fetchers.news import fetch_news

---
## Happy path — benign headlines → PASS

Positive, routine news from HIGH/MEDIUM sources. Expect `passed=True`, `threat_detected=False`.

In [10]:
candidate = {'ticker': 'NVDA', 'company_name': 'NVIDIA Corporation'}

benign = [
    {'headline': 'NVIDIA unveils next-gen GPU lineup at annual conference',
     'source': 'Reuters', 'reliability': 'HIGH', 'summary': ''},
    {'headline': 'Analysts raise NVIDIA price target on strong data-center demand',
     'source': 'CNBC', 'reliability': 'MEDIUM', 'summary': ''},
]

assess_gate2_news_threat(candidate, headlines=benign)

[gate2] NVDA: passed — no threat across 2 headlines


{'passed': True,
 'threat_detected': False,
 'threat_categories': [],
 'reason': 'NONE',
 'headlines_used': 2}

---
## Variation — fraud headlines → BLOCK

SEC fraud + short-seller attack from HIGH sources. Expect `passed=False`, `threat_detected=True`.

In [11]:
fraud = [
    {'headline': 'SEC charges NVIDIA executives with accounting fraud, opens formal investigation',
     'source': 'Bloomberg', 'reliability': 'HIGH', 'summary': ''},
    {'headline': 'Short seller alleges NVIDIA overstated revenue in scathing report',
     'source': 'Reuters', 'reliability': 'HIGH', 'summary': ''},
]

assess_gate2_news_threat(candidate, headlines=fraud)

[gate2] NVDA: BLOCKED — fraud_accounting_sec, short_seller_activist: SEC formal investigation into accounting fraud by NVIDIA executives (HIGH-reliability Bloomberg source) and a HIGH-reliability short-seller report alleging revenue overstatement both constitute catastrophic threats.


{'passed': False,
 'threat_detected': True,
 'threat_categories': ['fraud_accounting_sec', 'short_seller_activist'],
 'reason': 'SEC formal investigation into accounting fraud by NVIDIA executives (HIGH-reliability Bloomberg source) and a HIGH-reliability short-seller report alleging revenue overstatement both constitute catastrophic threats.',
 'headlines_used': 2}

---
## Real news — multiple tickers

The genuine end-to-end path (live `fetch_news` + live LLM), not synthetic fixtures. Three tickers across different sectors — Tesla (auto/tech), Apple (consumer tech), Pfizer (pharma) — so you can see real variation in what the gate flags. Results change with the news window.

In [12]:
# Real news across multiple tickers — the pipeline shape: the caller fetches once per ticker,
# then passes the headlines into the gate. Live API + live LLM, so verdicts vary with the news window.
for tkr, name in [('TSLA', 'Tesla, Inc.'),
                  ('AAPL', 'Apple Inc.'),
                  ('PFE',  'Pfizer Inc.')]:
    headlines = fetch_news(tkr) or []
    r = assess_gate2_news_threat({'ticker': tkr, 'company_name': name}, headlines)
    verdict = 'PASS' if r['passed'] else f'BLOCK ({", ".join(r["threat_categories"])})'
    print(f"{tkr:<5} {verdict:<30} headlines={r['headlines_used']}  reason={r['reason']}")

[gate2] TSLA: passed — no threat across 5 headlines
TSLA  PASS                           headlines=5  reason=NONE
[gate2] AAPL: passed — no threat across 5 headlines
AAPL  PASS                           headlines=5  reason=NONE
[gate2] PFE: passed — no threat across 5 headlines
PFE   PASS                           headlines=5  reason=NONE


---
## Failure / edge — empty headlines → pass, no LLM call

No news to assess means no threat. Returns immediately without spending a Claude call.

In [5]:
assess_gate2_news_threat(candidate, headlines=[])

[gate2] NVDA: no headlines — passing (no threat)


{'passed': True,
 'threat_detected': False,
 'threat_categories': [],
 'reason': 'NONE',
 'headlines_used': 0}

---
## Free-play

Try your own headlines, tickers, or edge cases below.